# Reuse a CLI login session → OCAPI

Reuses the token the B2C CLI stored in the shared `auth-sessions.json` — no
secret in `dw.json`. **Log in once first:**

```
b2c auth login <your clientId>
```

This finds that session by `clientId` and uses it.

In [ ]:
import json
from pathlib import Path

from b2c_tooling_sdk import CreateB2CInstanceOptions, ResolveConfigOptions, list_code_versions, resolve_config
from b2c_tooling_sdk.auth import StatefulOAuthStrategy, StatefulOAuthStrategyOptions, find_auth_session

DW_JSON = Path("../dw.json").resolve()
raw = json.loads(DW_JSON.read_text())
client_id = raw["clientId"]
account_manager_host = raw.get("accountManagerHost") or "account.demandware.com"

session = find_auth_session(client_id)
assert session is not None, f"No stored session. Run: b2c auth login {client_id}"
strategy = StatefulOAuthStrategy(session, StatefulOAuthStrategyOptions(account_manager_host))
config = await resolve_config(options=ResolveConfigOptions(config_path=str(DW_JSON)))
instance = config.create_b2c_instance(CreateB2CInstanceOptions(oauth_strategy=strategy))

In [ ]:
versions = await list_code_versions(instance)
for version in versions:
    print(version.id, "(active)" if version.active else "")